# Notebook 08 — Risk and factor analytics

**Prerequisites:** complete **notebook 07** in the analytics curriculum before this one.

**In this notebook:** regression-style **factor exposures** (beta, multi-factor alphas), **tail risk** (VaR, expected shortfall, higher moments), **relative performance** versus a benchmark (capture, tracking error, information ratio), **ruin** simulation, and **drawdown episode** detail — all via `finstack.analytics` Rust-backed bindings.


## Risk analytics and factor models

**Factor models** explain asset or fund returns as a linear combination of systematic factors plus idiosyncratic noise. A single-index model uses one benchmark (often “the market”); multi-factor models add size, value, momentum, or custom factors. Estimated **betas** are exposures; the intercept is **alpha** (excess return not explained by factors), and **R²** measures how much variance the factors explain.

**Risk analytics** complements that regression view with **distribution tails** (historical VaR, parametric Gaussian VaR, Cornish–Fisher which adjusts for skew and kurtosis, and expected shortfall), **benchmark-relative behaviour** (how much you participate in up markets vs down markets), and **path risk** (probability of breaching a wealth or drawdown constraint, and the shape of historical drawdowns).

Together, these tools support the same questions portfolio managers and risk officers ask: *What am I exposed to? How bad can short-term losses get? How do I behave versus the benchmark? What are the deepest past drawdowns?*


## API walkthrough — factor analysis

`beta` runs a dedicated beta regression with a standard error and asymptotic confidence band. `greeks` returns **annualized alpha**, beta, and R² for a single benchmark. `multi_factor_greeks` generalizes to several aligned return series (same length as the fund).

Below, `confidence` for VaR-style helpers is expressed as a **confidence level** (e.g. `0.95` → 95% VaR, i.e. the loss threshold associated with the worst 5% outcomes for historical VaR).


In [ ]:
from finstack.analytics import beta, greeks, multi_factor_greeks

fund_returns = [0.002, -0.001, 0.004, 0.001, -0.002, 0.003, 0.0, 0.0015]
bench_returns = [0.0015, -0.0005, 0.003, 0.0005, -0.0015, 0.0025, -0.0005, 0.001]
mkt_rets = bench_returns
smb_rets = [0.0005, 0.0002, -0.0008, 0.0004, -0.0003, 0.0006, 0.0001, -0.0002]
hml_rets = [-0.0002, 0.0003, 0.0001, -0.0001, 0.0002, -0.0004, 0.0002, 0.0001]

beta_result = beta(fund_returns, bench_returns)
print(f"Beta: {beta_result.beta:.4f}")
print(f"Std err: {beta_result.std_err:.4f}")
print(f"95% CI: [{beta_result.ci_lower:.4f}, {beta_result.ci_upper:.4f}]")

greeks_result = greeks(fund_returns, bench_returns)
print(f"Alpha (ann.): {greeks_result.alpha:.4f}")
print(f"Beta: {greeks_result.beta:.4f}")
print(f"R-squared: {greeks_result.r_squared:.4f}")

mf = multi_factor_greeks(fund_returns, [mkt_rets, smb_rets, hml_rets])
print(f"Multi-factor alpha (ann.): {mf.alpha:.4f}")
print(f"Betas: {mf.betas}")
print(f"R-squared: {mf.r_squared:.4f}")
print(f"Adj R-squared: {mf.adjusted_r_squared:.4f}")
print(f"Residual vol: {mf.residual_vol:.4f}")


### Rolling greeks, Treynor, and M²

`rolling_greeks` fits the single-index model on a moving window and returns time series of **alpha** and **beta** (aligned to `dates`). **Treynor** and **M²** are classic summary ratios computed from **annualized** return and risk inputs together with benchmark volatility or beta.

In [ ]:
import datetime
import math
import statistics

from finstack.analytics import greeks, m_squared, rolling_greeks, treynor

fund_returns = [0.0012 * ((i % 11) - 5) / 10 for i in range(90)]
bench_returns = [0.001 * ((i % 9) - 4) / 10 for i in range(90)]
dates = [datetime.date(2024, 1, 2) + datetime.timedelta(days=i) for i in range(90)]

rg = rolling_greeks(fund_returns, bench_returns, dates, window=30)
print(f"Rolling windows: {len(rg.alphas)}; last alpha: {rg.alphas[-1]:.6f}; last beta: {rg.betas[-1]:.4f}")

gr = greeks(fund_returns, bench_returns)
ann_factor = 252.0
mu_f = statistics.mean(fund_returns) * ann_factor
mu_b = statistics.mean(bench_returns) * ann_factor
vol_f = statistics.stdev(fund_returns) * math.sqrt(ann_factor)
vol_b = statistics.stdev(bench_returns) * math.sqrt(ann_factor)
rf = 0.04

t = treynor(mu_f, rf, gr.beta)
m2 = m_squared(mu_f, vol_f, vol_b, rf)
print(f"Treynor (from sample means & greeks beta): {t:.6f}")
print(f"M² (from sample ann. vols): {m2:.6f}")

### VaR and expected shortfall

**Historical VaR** uses the empirical quantile of past returns. **Parametric VaR** assumes Gaussian returns. **Cornish–Fisher VaR** adjusts the Gaussian quantile using sample skewness and kurtosis. **Expected shortfall** (CVaR) averages losses beyond the VaR threshold — a coherent tail measure.


In [ ]:
from finstack.analytics import value_at_risk, expected_shortfall, parametric_var, cornish_fisher_var

returns = [-0.04, -0.02, -0.015, -0.01, -0.005, 0.0, 0.002, 0.005, 0.008, 0.012]
conf = 0.95

var_hist = value_at_risk(returns, conf)
es_hist = expected_shortfall(returns, conf)
var_param = parametric_var(returns, conf)
var_cf = cornish_fisher_var(returns, conf)

print(f"Historical VaR ({conf:.0%}): {var_hist:.6f}")
print(f"Parametric VaR ({conf:.0%}): {var_param:.6f}")
print(f"Cornish-Fisher VaR ({conf:.0%}): {var_cf:.6f}")
print(f"Expected shortfall ({conf:.0%}): {es_hist:.6f}")


### Capture ratios and benchmark diagnostics

**Up-capture** and **down-capture** summarize how the fund moves when the benchmark is positive vs negative. The **capture ratio** combines them. **Tracking error** and the **information ratio** describe active risk and active return per unit of risk. **R²** and **batting average** add explanatory power and “win rate” versus the benchmark.


In [ ]:
from finstack.analytics import (
    up_capture,
    down_capture,
    capture_ratio,
    tracking_error,
    information_ratio,
    r_squared,
    batting_average,
)

fund_returns = [0.012, -0.008, 0.006, -0.003, 0.009, -0.011, 0.004, 0.002]
bench_returns = [0.010, -0.009, 0.005, -0.004, 0.007, -0.010, 0.003, 0.001]

print(f"Up capture: {up_capture(fund_returns, bench_returns):.4f}")
print(f"Down capture: {down_capture(fund_returns, bench_returns):.4f}")
print(f"Capture ratio: {capture_ratio(fund_returns, bench_returns):.4f}")
print(f"Tracking error: {tracking_error(fund_returns, bench_returns):.6f}")
print(f"Information ratio: {information_ratio(fund_returns, bench_returns):.4f}")
print(f"R-squared: {r_squared(fund_returns, bench_returns):.4f}")
print(f"Batting average: {batting_average(fund_returns, bench_returns):.4f}")


### Higher moments and tail diagnostics

**Skewness** and **excess kurtosis** describe asymmetry and fat tails. **Tail ratio** contrasts upper and lower tail magnitudes; **outlier win/loss ratios** focus on extreme positive vs negative days.


In [ ]:
from finstack.analytics import skewness, kurtosis, tail_ratio, outlier_win_ratio, outlier_loss_ratio

returns = [-0.06, -0.02, -0.01, 0.0, 0.005, 0.01, 0.02, 0.07]

print(f"Skewness: {skewness(returns):.4f}")
print(f"Excess kurtosis: {kurtosis(returns):.4f}")
print(f"Tail ratio: {tail_ratio(returns):.4f}")
print(f"Outlier win ratio: {outlier_win_ratio(returns):.4f}")
print(f"Outlier loss ratio: {outlier_loss_ratio(returns):.4f}")


### Ruin probability (Monte Carlo)

`estimate_ruin` bootstraps historical returns into forward paths and estimates the probability of hitting a **ruin event** — for example, wealth falling below a fraction of initial capital. `RuinModel` controls horizon, number of paths, block size, and RNG seed; the result includes a Monte Carlo standard error and confidence interval.


In [ ]:
from finstack.analytics import RuinDefinition, RuinModel, estimate_ruin

returns = [0.001 * (i % 7 - 3) for i in range(200)]
ruin_def = RuinDefinition.wealth_floor(0.85)
ruin_model = RuinModel(horizon_periods=126, n_paths=4_000, seed=42)
ruin_est = estimate_ruin(returns, ruin_def, ruin_model)

print(f"Ruin probability: {ruin_est.probability:.4f}")
print(f"Std error: {ruin_est.std_err:.4f}")
print(f"95% CI: [{ruin_est.ci_lower:.4f}, {ruin_est.ci_upper:.4f}]")


### Drawdown episodes

`to_drawdown_series` converts simple returns into a running drawdown path (non-positive). `drawdown_details` takes that path plus **aligned calendar dates** and returns the deepest episodes with start, valley, and recovery dates. Compose `max_drawdown(to_drawdown_series(returns))` for the single worst peak-to-trough depth.


In [ ]:
import datetime
from finstack.analytics import drawdown_details, to_drawdown_series, max_drawdown

returns = [0.03, 0.01, -0.04, -0.02, 0.015, 0.02, -0.01, 0.005]
start = datetime.date(2024, 1, 2)
dates = []
d = start
for _ in returns:
    while d.weekday() >= 5:
        d += datetime.timedelta(days=1)
    dates.append(d)
    d += datetime.timedelta(days=1)

dd_path = to_drawdown_series(returns)
mdd = max_drawdown(dd_path)
print(f"Max drawdown from returns: {mdd:.2%}")

episodes = drawdown_details(dd_path, dates, n=3)
for i, ep in enumerate(episodes, start=1):
    end = ep.end()
    end_s = end.isoformat() if end else "(ongoing)"
    print(
        f"  #{i} Start: {ep.start().isoformat()}, Valley: {ep.valley().isoformat()}, "
        f"End: {end_s}, Depth: {ep.max_drawdown:.2%}"
    )


## Mini-example: hedge fund risk report

We simulate **252 daily** fund returns with mild **alpha** and exposures to a **market** factor plus **size** and **value** style factors, then print a compact risk report: regressions, tail measures, benchmark behaviour, ruin probability, and the **top three** historical drawdown episodes.


In [ ]:
import datetime
import random

from finstack.analytics import (
    batting_average,
    beta,
    capture_ratio,
    cornish_fisher_var,
    down_capture,
    drawdown_details,
    estimate_ruin,
    expected_shortfall,
    greeks,
    information_ratio,
    max_drawdown,
    multi_factor_greeks,
    parametric_var,
    r_squared,
    RuinDefinition,
    RuinModel,
    to_drawdown_series,
    tracking_error,
    up_capture,
    value_at_risk,
)

random.seed(42)
n = 252

mkt_returns = [random.gauss(0.0, 0.008) for _ in range(n)]
factor2_returns = [random.gauss(0.0, 0.005) for _ in range(n)]
factor3_returns = [random.gauss(0.0, 0.004) for _ in range(n)]

fund_returns = []
for i in range(n):
    noise = random.gauss(0.0, 0.0035)
    r = (
        0.00025
        + 0.88 * mkt_returns[i]
        + 0.12 * factor2_returns[i]
        - 0.06 * factor3_returns[i]
        + noise
    )
    fund_returns.append(r)

start = datetime.date(2024, 1, 2)
dates = []
d = start
for _ in range(n):
    while d.weekday() >= 5:
        d += datetime.timedelta(days=1)
    dates.append(d)
    d += datetime.timedelta(days=1)

print("=== Synthetic data ===")
print(f"Days: {n}; seed=42; fund mean daily return: {sum(fund_returns)/n:.6f}")
print(f"Benchmark (market) mean daily return: {sum(mkt_returns)/n:.6f}")
print(f"Max drawdown (fund): {max_drawdown(to_drawdown_series(fund_returns)):.2%}")


In [ ]:
# 1) Single-factor regression (market): beta CI + greeks alpha/R²

beta_result = beta(fund_returns, mkt_returns)
gr = greeks(fund_returns, mkt_returns)

print("=== 1) Single-factor vs market ===")
print(f"beta: {beta_result.beta:.4f} (SE {beta_result.std_err:.4f})")
print(f"95% CI: [{beta_result.ci_lower:.4f}, {beta_result.ci_upper:.4f}]")
print(f"greeks alpha (ann.): {gr.alpha:.6f}; beta: {gr.beta:.4f}; R²: {gr.r_squared:.4f}")


In [ ]:
# 2) Three-factor regression

mf = multi_factor_greeks(fund_returns, [mkt_returns, factor2_returns, factor3_returns])

print("=== 2) Three-factor model ===")
print(f"Alpha (ann.): {mf.alpha:.6f}")
print(f"Betas [mkt, size, value]: {[round(b, 4) for b in mf.betas]}")
print(f"R²: {mf.r_squared:.4f}; Adj R²: {mf.adjusted_r_squared:.4f}")
print(f"Residual vol: {mf.residual_vol:.6f}")


In [ ]:
# 3) VaR comparison + 4) Expected shortfall

conf = 0.95
var_hist = value_at_risk(fund_returns, conf)
var_param = parametric_var(fund_returns, conf)
var_cf = cornish_fisher_var(fund_returns, conf)
es_hist = expected_shortfall(fund_returns, conf)

print("=== 3) VaR comparison (daily, same confidence) ===")
print(f"Confidence level: {conf:.0%}")
print(f"Historical VaR:   {var_hist:.6f}")
print(f"Parametric VaR:   {var_param:.6f}")
print(f"Cornish-Fisher:   {var_cf:.6f}")
print("=== 4) Expected shortfall ===")
print(f"Expected shortfall: {es_hist:.6f}")


In [ ]:
# 5) Capture ratios vs market benchmark

uc = up_capture(fund_returns, mkt_returns)
dc = down_capture(fund_returns, mkt_returns)
cr = capture_ratio(fund_returns, mkt_returns)
te = tracking_error(fund_returns, mkt_returns)
ir = information_ratio(fund_returns, mkt_returns)
rsq = r_squared(fund_returns, mkt_returns)
ba = batting_average(fund_returns, mkt_returns)

print("=== 5) Versus market benchmark ===")
print(f"Up capture: {uc:.4f}; Down capture: {dc:.4f}; Capture ratio: {cr:.4f}")
print(f"Tracking error: {te:.6f}; Information ratio: {ir:.4f}")
print(f"R²: {rsq:.4f}; Batting average: {ba:.4f}")


In [ ]:
# 6) Ruin probability (per curriculum snippet; may be near zero with this simulation)

ruin_def = RuinDefinition.wealth_floor(0.5)
ruin_model = RuinModel(horizon_periods=252, n_paths=10_000, seed=42)
ruin_est = estimate_ruin(fund_returns, ruin_def, ruin_model)

print("=== 6) Ruin estimate (wealth floor 50% of initial) ===")
print(f"Ruin probability: {ruin_est.probability:.4f}")
print(f"Std error: {ruin_est.std_err:.4f}")
print(f"95% CI: [{ruin_est.ci_lower:.4f}, {ruin_est.ci_upper:.4f}]")


In [ ]:
# 7) Top 3 drawdown episodes

dd_path = to_drawdown_series(fund_returns)
episodes = drawdown_details(dd_path, dates, n=3)

print("=== 7) Top drawdown episodes ===")
for i, ep in enumerate(episodes, start=1):
    end = ep.end()
    end_s = end.isoformat() if end else "(ongoing)"
    print(
        f"  #{i} Start: {ep.start().isoformat()}, Valley: {ep.valley().isoformat()}, "
        f"End: {end_s}, Depth: {ep.max_drawdown:.2%}"
    )


## Takeaways

- **Factor analytics** — `greeks` and `multi_factor_greeks` give you interpretable **alpha**, **betas**, and **fit quality**; `beta` adds **uncertainty** around market beta.
- **Tail risk** — compare **historical**, **Gaussian**, and **Cornish–Fisher** VaR to see how much distribution shape matters; pair VaR with **expected shortfall** for average loss beyond the threshold.
- **Benchmark lens** — **capture** ratios, **tracking error**, and **information ratio** summarize how a fund behaves **relative** to its bogey, not just its standalone volatility.
- **Path risk** — **ruin** estimation stress-tests wealth constraints under resampled returns; **drawdown episodes** localize the worst peak-to-trough experiences in calendar time.

Re-run with your own return series (and aligned factor columns) to move from synthetic demos to production-style monitoring.
